# 03 - Preprocessing & HOG Feature Engineering

Project: **RoadEye - Traffic Sign Recognition Using HOG Features and SVM** (Machine Learning Innovative Assignment).

> Run cells top to bottom. All numbers are computed when the cell runs; nothing is typed in by hand.

In [ ]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
from src import config, pipeline as P
np.random.seed(config.SEED)                      # reproducibility
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("RUN MODE:", config.RUN_MODE, "| seed:", config.SEED)

In [ ]:
def show(name, width=900):
    """Display a saved figure from results/figures/."""
    display(Image(filename=str(config.FIGURES_DIR / name), width=width))

def metrics(name):
    return json.loads((config.METRICS_DIR / name).read_text())

## 1. What preprocessing does to one image
Original (RGB, variable size) -> ROI crop -> grayscale -> 32x32 -> HOG.

In [ ]:
data = P.stage_prepare(save_figures=True)
from src.data_loader import read_image_rgb
from src.preprocessing import preprocess_image
i = data.idx('train')[0]
row = data.meta.iloc[i]
raw_img = read_image_rgb(row['path'])
roi = (row.roi_x1, row.roi_y1, row.roi_x2, row.roi_y2)
proc = preprocess_image(raw_img, roi)
fig, ax = plt.subplots(1, 2, figsize=(5, 2.6)); ax[0].imshow(raw_img); ax[0].set_title(f'original {raw_img.shape[1]}x{raw_img.shape[0]}')
ax[1].imshow(proc, cmap='gray'); ax[1].set_title('preprocessed 32x32'); [a.axis('off') for a in ax]; plt.show()

## 2. HOG: parameters and dimensions
HOG describes local edge *orientation* statistics, which capture the sign's outline and symbol shape and are largely invariant to illumination (gamma compression + block L2-Hys normalisation).

In [ ]:
from src.hog_features import hog_feature_dim
print('HOG parameters:', config.HOG_PARAMS)
print('raw-pixel dimension :', data.raw.shape[1])
print('HOG dimension       :', data.hog.shape[1], '(analytical:', hog_feature_dim(), ')')
show('hog_visualization.png', 900)

## 3. Do the representations separate the classes?

In [ ]:
show('raw_vs_hog_features.png', 900)

## 4. Feature scaling without leakage
The `StandardScaler` is **fitted on the training split only**, then applied to validation (and, later, test).

In [ ]:
from src.preprocessing import fit_scaler
Xtr, Xva = data.X('hog', 'train'), data.X('hog', 'val')
sc = fit_scaler(Xtr)
print('train mean after scaling (should be ~0):', float(sc.transform(Xtr).mean().round(6)))
print('val   mean after scaling (NOT forced to 0):', float(sc.transform(Xva).mean().round(6)))
np.save(config.PROCESSED_DIR / 'hog_train_unscaled.npy', Xtr)  # processed features saved for reuse

**Feature selection decision:** none applied. All 1764 HOG dimensions are kept; dimensionality reduction (PCA) was not required for the chosen models and any such step would have to be fitted on training data only.